# TS-models + PMSD -- SSD-truncated real-life logs

In [ ]:
import sys
import json
import time
from pathlib import Path

import pandas as pd
import numpy as np
import pm4py

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "steady_state_detection"))
sys.path.insert(0, str(ROOT / "analysis"))
sys.path.insert(0, str(ROOT / "plain-field"))

REAL_DATA_DIR = ROOT / "data" / "real-life"
BEST_MODELS = ROOT / "best_models"
RESULTS = ROOT / "results"
TRIM_NAME = "ssd"
SSD_TRIM_CFG = {"method": "ssd", "frac": 0.6, "k": 1.5, "pct": 0.25}

REAL_DATASETS = ["bpic12-a", "bpic15-1", "bpic15-2", "bpic17-o",
                "bpic20-dom", "bpic20-int", "helpdesk", "sepsis"]

print(f"ROOT = {ROOT}")
print(f"{len(REAL_DATASETS)} real-life datasets")

In [ ]:
def load_data_for_ssd(xes_path):
    """Returns (df, cc, tt, train_df, val_df, test_df) for the ssd trim."""
    from time_series_preprocessing import Split3WayConfig, split_timeseries
    from time_series_creation import create_concurrent_cases_timeseries, create_avg_throughtput_time_timeseries
    from create_prefixes_from_windows import make_three_way_split
    from ssd_trim import run_ssd_trim

    log = pm4py.read_xes(str(xes_path))

    full_cc_raw = create_concurrent_cases_timeseries(log, plot=False)
    ssd_result = run_ssd_trim(log, window_step="D")
    canonical_end = ssd_result["cutoff"] if ssd_result["cutoff"] is not None else full_cc_raw.index[-1]
    full_cc_trimmed = full_cc_raw[full_cc_raw.index <= canonical_end]
    split_cfg = Split3WayConfig(train_frac=0.70, val_frac=0.10, test_frac=0.20)
    _, _, _, train_split, val_split = split_timeseries(full_cc_trimmed, split_cfg)

    full_tt_raw = create_avg_throughtput_time_timeseries(log, plot=False)
    full_tt_trimmed = full_tt_raw[full_tt_raw.index <= canonical_end]

    def _slice(raw, trimmed):
        idx = trimmed.index
        lo = train_split.tz_convert(None) if idx.tz is None else train_split
        hi = val_split.tz_convert(None) if idx.tz is None else val_split
        return {
            "raw": raw, "trimmed": trimmed,
            "train": trimmed[idx <= lo],
            "val": trimmed[(idx > lo) & (idx <= hi)],
            "test": trimmed[idx > hi],
            "train_split": train_split, "val_split": val_split,
        }

    cc = _slice(full_cc_raw, full_cc_trimmed)
    tt = _slice(full_tt_raw, full_tt_trimmed)

    df = pm4py.convert_to_dataframe(log)
    df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
    df = df.dropna(subset=["case:concept:name"])
    _cols = {"case:concept:name": "caseid", "concept:name": "task",
             "lifecycle:transition": "event_type", "time:timestamp": "end_timestamp"}
    _cols["org:resource" if "org:resource" in df.columns else "org:group"] = "user"
    df = df.rename(columns=_cols)
    df["task"] = df["task"].fillna("unk")
    df["user"] = df["user"].fillna("unk")

    train_, val_, test_ = make_three_way_split(
        df, case_col="caseid", time_col="end_timestamp",
        train_split=cc["train_split"], val_split=cc["val_split"], full_traces=True,
    )
    return df, cc, tt, train_, val_, test_

In [ ]:
PMSD_MAIN = ROOT / "simulation_baselines" / "PMSD-main"
sys.path.insert(0, str(PMSD_MAIN))
from pmsd import run_pmsd

def _pmsd_done(name):
    p = RESULTS / TRIM_NAME / name / f"metrics_{name}_pmsd.csv"
    return p.exists()

for name in REAL_DATASETS:
    print(f"\n{'='*60}\n{name} (PMSD)\n{'='*60}")
    if _pmsd_done(name):
        print("  [skip] metrics already exist")
        continue
    full_log = pm4py.read_xes(str(REAL_DATA_DIR / f"{name}.xes"))
    _, cc, tt, _, _, _ = load_data_for_ssd(REAL_DATA_DIR / f"{name}.xes")
    try:
        res = run_pmsd(name, full_log, trim=TRIM_NAME, is_real=True, save=True,
                       precomputed_splits={"concurrent_cases": cc, "throughput_time": tt})
        print(f"  {res['metrics']}")
    except Exception as e:
        print(f"  FAILED: {e}")

In [ ]:
rows = []
for name in REAL_DATASETS:
    for p in [
        RESULTS / TRIM_NAME / name / f"metrics_{name}.csv",
        RESULTS / TRIM_NAME / name / f"metrics_{name}_pmsd.csv",
    ]:
        if p.exists():
            rows.append(pd.read_csv(p))

if rows:
    all_metrics = pd.concat(rows, ignore_index=True)
    display(all_metrics.pivot_table(index=["dataset", "model"], columns="series", values="mae"))
else:
    print("No results yet.")